# UPI Transaction Intelligence & Fraud Analytics — 2024
### IBM SkillsBuild Data Analytics with AI Academic Internship

**Author:** Sairam Ranveerkar  
**Dataset:** 2024 UPI transaction dataset  
**Objective:** Analyze payment performance, customer behavior, banking/merchant trends, fraud-flagged transactions, and potential transaction-volume anomalies.

> **Interpretation note:** `fraud_flag = 1` is a dataset-provided flag. It is reported as a *fraud-flagged transaction*, not as independently confirmed fraud.


## 1. Setup


In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_FILE = Path("upi_transactions_2024_clean.csv")
OUTPUT_DIR = Path("analysis_outputs")
CHART_DIR = OUTPUT_DIR / "charts"
OUTPUT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)

print("Working directory:", Path.cwd())
print("Dataset:", DATA_FILE)


## 2. Load and validate the dataset


In [ ]:
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

print("Shape:", df.shape)
display(df.head())


In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate transaction IDs:", int(df["transaction_id"].duplicated().sum()))
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))


## 3. KPI summary


In [ ]:
total_transactions = len(df)
total_value = df["amount_inr"].sum()
average_amount = df["amount_inr"].mean()
median_amount = df["amount_inr"].median()
successful = (df["transaction_status"].str.upper() == "SUCCESS").sum()
failed = (df["transaction_status"].str.upper() == "FAILED").sum()
fraud_flagged = df["fraud_flag"].sum()
fraud_flagged_amount = df.loc[df["fraud_flag"] == 1, "amount_inr"].sum()

kpi = pd.DataFrame({
    "Metric": [
        "Total Transactions", "Total Transaction Value (INR)",
        "Average Transaction (INR)", "Median Transaction (INR)",
        "Minimum Transaction (INR)", "Maximum Transaction (INR)",
        "Successful Transactions", "Failed Transactions",
        "Success Rate (%)", "Failure Rate (%)",
        "Fraud-Flagged Transactions", "Fraud-Flag Rate (%)",
        "Fraud-Flagged Amount (INR)"
    ],
    "Value": [
        total_transactions, total_value, average_amount, median_amount,
        df["amount_inr"].min(), df["amount_inr"].max(),
        successful, failed, successful/total_transactions*100,
        failed/total_transactions*100, fraud_flagged,
        fraud_flagged/total_transactions*100, fraud_flagged_amount
    ]
})
display(kpi)
kpi.to_csv(OUTPUT_DIR / "kpi_summary.csv", index=False)


## 4. Monthly transaction analysis


In [ ]:
monthly = df.groupby("year_month", sort=True).agg(
    transactions=("transaction_id","count"),
    transaction_value_inr=("amount_inr","sum"),
    average_amount_inr=("amount_inr","mean"),
    failed_transactions=("transaction_status", lambda s: (s.str.upper()=="FAILED").sum()),
    fraud_flags=("fraud_flag","sum")
).reset_index()
monthly["failure_rate_pct"] = monthly["failed_transactions"]/monthly["transactions"]*100
monthly["fraud_flag_rate_pct"] = monthly["fraud_flags"]/monthly["transactions"]*100
display(monthly)
monthly.to_csv(OUTPUT_DIR / "monthly_analysis.csv", index=False)


In [ ]:
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly, x="year_month", y="transactions", marker="o")
plt.title("Monthly UPI Transaction Volume — 2024")
plt.xlabel("Month")
plt.ylabel("Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(CHART_DIR / "monthly_transaction_volume.png", dpi=160)
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly, x="year_month", y="transaction_value_inr", marker="o")
plt.title("Monthly UPI Transaction Value — 2024")
plt.xlabel("Month")
plt.ylabel("Transaction Value (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(CHART_DIR / "monthly_transaction_value.png", dpi=160)
plt.show()


## 5. Transaction type and merchant analysis


In [ ]:
def grouped_analysis(column):
    out = df.groupby(column).agg(
        transactions=("transaction_id","count"),
        transaction_value_inr=("amount_inr","sum"),
        average_amount_inr=("amount_inr","mean"),
        failed_transactions=("transaction_status", lambda s: (s.str.upper()=="FAILED").sum()),
        fraud_flags=("fraud_flag","sum")
    ).reset_index()
    out["failure_rate_pct"] = out["failed_transactions"]/out["transactions"]*100
    out["fraud_flag_rate_pct"] = out["fraud_flags"]/out["transactions"]*100
    return out

transaction_type = grouped_analysis("transaction_type")
merchant = grouped_analysis("merchant_category")

display(transaction_type.sort_values("transaction_value_inr", ascending=False))
display(merchant.sort_values("transaction_value_inr", ascending=False))

transaction_type.to_csv(OUTPUT_DIR / "transaction_type_analysis.csv", index=False)
merchant.to_csv(OUTPUT_DIR / "merchant_analysis.csv", index=False)


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=transaction_type.sort_values("transactions", ascending=False),
            x="transaction_type", y="transactions")
plt.title("Transaction Volume by Type")
plt.xlabel("Transaction Type")
plt.ylabel("Transactions")
plt.tight_layout()
plt.savefig(CHART_DIR / "transaction_type_volume.png", dpi=160)
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
top_m = merchant.sort_values("transactions", ascending=False)
sns.barplot(data=top_m, x="transactions", y="merchant_category")
plt.title("Transaction Volume by Merchant Category")
plt.xlabel("Transactions")
plt.ylabel("Merchant Category")
plt.tight_layout()
plt.savefig(CHART_DIR / "merchant_category_volume.png", dpi=160)
plt.show()


## 6. Regional and banking analysis


In [ ]:
state = grouped_analysis("sender_state")
sender_bank = grouped_analysis("sender_bank")
receiver_bank = grouped_analysis("receiver_bank")

display(state.sort_values("transaction_value_inr", ascending=False).head(10))
display(sender_bank.sort_values("transaction_value_inr", ascending=False))
display(receiver_bank.sort_values("transaction_value_inr", ascending=False))

state.to_csv(OUTPUT_DIR / "state_analysis.csv", index=False)
sender_bank.to_csv(OUTPUT_DIR / "sender_bank_analysis.csv", index=False)
receiver_bank.to_csv(OUTPUT_DIR / "receiver_bank_analysis.csv", index=False)


In [ ]:
top_states = state.sort_values("transaction_value_inr", ascending=False).head(10)
plt.figure(figsize=(10,6))
sns.barplot(data=top_states, x="transaction_value_inr", y="sender_state")
plt.title("Top States by Transaction Value")
plt.xlabel("Transaction Value (INR)")
plt.ylabel("Sender State")
plt.tight_layout()
plt.savefig(CHART_DIR / "state_transaction_value.png", dpi=160)
plt.show()


In [ ]:
top_banks = sender_bank.sort_values("transaction_value_inr", ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(data=top_banks, x="transaction_value_inr", y="sender_bank")
plt.title("Sender Banks by Transaction Value")
plt.xlabel("Transaction Value (INR)")
plt.ylabel("Sender Bank")
plt.tight_layout()
plt.savefig(CHART_DIR / "sender_bank_value.png", dpi=160)
plt.show()


## 7. Customer and usage behavior


In [ ]:
age_sender = grouped_analysis("sender_age_group")
age_receiver = grouped_analysis("receiver_age_group")
device = grouped_analysis("device_type")
network = grouped_analysis("network_type")

display(age_sender)
display(age_receiver)
display(device)
display(network)

age_sender.to_csv(OUTPUT_DIR / "sender_age_analysis.csv", index=False)
age_receiver.to_csv(OUTPUT_DIR / "receiver_age_analysis.csv", index=False)
device.to_csv(OUTPUT_DIR / "device_analysis.csv", index=False)
network.to_csv(OUTPUT_DIR / "network_analysis.csv", index=False)


In [ ]:
weekend = df.groupby("is_weekend").agg(
    transactions=("transaction_id","count"),
    transaction_value_inr=("amount_inr","sum"),
    average_amount_inr=("amount_inr","mean"),
    fraud_flags=("fraud_flag","sum")
).reset_index()
weekend["fraud_flag_rate_pct"] = weekend["fraud_flags"]/weekend["transactions"]*100
display(weekend)


## 8. Time-of-day and day-of-week analysis


In [ ]:
hourly = grouped_analysis("hour_of_day")
dow = grouped_analysis("day_of_week")
hourly.to_csv(OUTPUT_DIR / "hourly_analysis.csv", index=False)
dow.to_csv(OUTPUT_DIR / "day_of_week_analysis.csv", index=False)

display(hourly.sort_values("hour_of_day"))
display(dow.sort_values("transactions", ascending=False))


In [ ]:
plt.figure(figsize=(12,5))
sns.lineplot(data=hourly.sort_values("hour_of_day"),
             x="hour_of_day", y="transactions", marker="o")
plt.title("UPI Transaction Activity by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Transactions")
plt.tight_layout()
plt.savefig(CHART_DIR / "hourly_activity.png", dpi=160)
plt.show()


## 9. Fraud-flag analytics


In [ ]:
fraud_summary = pd.DataFrame({
    "metric": [
        "Fraud-flagged transactions",
        "Fraud-flagged amount (INR)",
        "Fraud-flag rate (%)"
    ],
    "value": [
        int(df["fraud_flag"].sum()),
        int(df.loc[df["fraud_flag"] == 1, "amount_inr"].sum()),
        df["fraud_flag"].mean()*100
    ]
})
display(fraud_summary)


In [ ]:
fraud_by_state = state.sort_values("fraud_flag_rate_pct", ascending=False)
fraud_by_type = transaction_type.sort_values("fraud_flag_rate_pct", ascending=False)
fraud_by_device = device.sort_values("fraud_flag_rate_pct", ascending=False)
fraud_by_network = network.sort_values("fraud_flag_rate_pct", ascending=False)

display(fraud_by_state.head(10))
display(fraud_by_type)
display(fraud_by_device)
display(fraud_by_network)


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=fraud_by_state.head(10),
            x="fraud_flag_rate_pct", y="sender_state")
plt.title("Top States by Fraud-Flag Rate")
plt.xlabel("Fraud-Flag Rate (%)")
plt.ylabel("Sender State")
plt.tight_layout()
plt.savefig(CHART_DIR / "fraud_rate_by_state.png", dpi=160)
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=hourly.sort_values("fraud_flags", ascending=False).head(10),
            x="hour_of_day", y="fraud_flags")
plt.title("Hours with the Most Fraud Flags")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud-Flagged Transactions")
plt.tight_layout()
plt.savefig(CHART_DIR / "fraud_flags_by_hour.png", dpi=160)
plt.show()


## 10. Daily volume anomaly screening


In [ ]:
daily = df.assign(date_only=df["timestamp"].dt.date).groupby("date_only").agg(
    transactions=("transaction_id","count"),
    transaction_value_inr=("amount_inr","sum"),
    fraud_flags=("fraud_flag","sum")
).reset_index()

daily["rolling_7d_mean"] = daily["transactions"].rolling(7, min_periods=3).mean()
daily["rolling_7d_std"] = daily["transactions"].rolling(7, min_periods=3).std()
daily["z_score"] = (
    daily["transactions"] - daily["rolling_7d_mean"]
) / daily["rolling_7d_std"].replace(0, np.nan)
daily["potential_volume_anomaly"] = daily["z_score"].abs() >= 2

display(daily.sort_values("z_score", key=lambda s: s.abs(), ascending=False).head(15))
daily.to_csv(OUTPUT_DIR / "daily_anomaly_analysis.csv", index=False)


In [ ]:
plt.figure(figsize=(13,5))
plt.plot(daily["date_only"], daily["transactions"], label="Daily transactions")
plt.plot(daily["date_only"], daily["rolling_7d_mean"], label="7-day rolling average")
anomalies = daily[daily["potential_volume_anomaly"]]
plt.scatter(anomalies["date_only"], anomalies["transactions"], label="Potential volume anomaly")
plt.title("Daily Transaction Volume and 7-Day Baseline")
plt.xlabel("Date")
plt.ylabel("Transactions")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(CHART_DIR / "daily_volume_anomalies.png", dpi=160)
plt.show()


## 11. Optional unsupervised amount-anomaly screening


In [ ]:
# Optional: screen unusually large/small transaction amounts.
# This is an anomaly screen, NOT a fraud classifier.
q1, q3 = df["amount_inr"].quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df["amount_anomaly_iqr"] = (df["amount_inr"] < lower) | (df["amount_inr"] > upper)
print("IQR lower bound:", lower)
print("IQR upper bound:", upper)
print("Amount anomalies:", int(df["amount_anomaly_iqr"].sum()))


## 12. Export final analytical tables


In [ ]:
# Save a compact executive summary for downstream dashboard/AI use.
executive = {
    "total_transactions": int(total_transactions),
    "total_transaction_value_inr": float(total_value),
    "average_transaction_inr": float(average_amount),
    "median_transaction_inr": float(median_amount),
    "successful_transactions": int(successful),
    "failed_transactions": int(failed),
    "success_rate_pct": float(successful/total_transactions*100),
    "failure_rate_pct": float(failed/total_transactions*100),
    "fraud_flagged_transactions": int(fraud_flagged),
    "fraud_flag_rate_pct": float(fraud_flagged/total_transactions*100),
    "fraud_flagged_amount_inr": float(fraud_flagged_amount)
}
with open(OUTPUT_DIR / "executive_summary.json", "w", encoding="utf-8") as f:
    json.dump(executive, f, indent=2)

print("Analysis completed successfully.")
print(executive)
